# MODELO BASADO EN CONTENIDO

Construir un sistema de recomendación basado en contenido que identifique películas similares utilizando características descriptivas. El modelo utiliza información de Gold ML, incluye géneros, palabras clave, director, actores, sinopsis e idioma.

Las características textuales son vectorizadas mediante TF-IDF y las películas similares serán identificadas mediante distancia coseno.

## 1. IMPORTACIONES

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

## 2. CONFIGURACIÓN DEL PROYECTO

In [2]:
PROJECT_ROOT = Path.cwd()

while (
    not (PROJECT_ROOT / "src").exists()
    and PROJECT_ROOT.parent != PROJECT_ROOT
):
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import (
    GOLD_ML_CONTENT_DIR,
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nGOLD_ML_CONTENT_DIR:")
print(GOLD_ML_CONTENT_DIR)

PROJECT_ROOT:
d:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch

GOLD_ML_CONTENT_DIR:
D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\gold\ml\content_based


## 3. Cargamos el dataset Content-Based

In [5]:
content = pd.read_parquet(
    GOLD_ML_CONTENT_DIR
    / "content_based_features.parquet"
)

## 4. Revisión basica

In [ ]:
print(
    "Películas:",
    content["movie_id"].nunique()
)

print(
    "Registros:",
    len(content)
)

content.isna().sum()

Películas: 87585
Registros: 87585


movie_id                      int64
tmdb_id                       Int64
title                        string
genres                       string
tmdb_genres                  object
keywords                     object
director                     string
actors                       object
overview                     string
original_language            string
release_date         datetime64[us]
dtype: object

## 5. Revisamos tipo de datos

In [16]:
for columna in [
    "genres",
    "tmdb_genres",
    "keywords",
    "actors",
]:
    if columna in content.columns:
        print("\n", columna)
        print(content[columna].head())


 genres
0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
Name: genres, dtype: string

 tmdb_genres
0    [Familia, Comedia, Animación, Aventura]
1              [Aventura, Fantasía, Familia]
2                         [Romance, Comedia]
3                  [Comedia, Drama, Romance]
4                         [Comedia, Familia]
Name: tmdb_genres, dtype: object

 keywords
0    [rescue, friendship, mission, jealousy, villai...
1    [based on novel or book, giant insect, board g...
2    [fishing, sequel, old man, best friend, weddin...
3    [based on novel or book, single mother, divorc...
4    [daughter, baby, parent child relationship, mi...
Name: keywords, dtype: object

 actors
0    [Tom Hanks, Tim Allen, Don Rickles, Jim Varney...
1    [Robin Williams, Kirsten Dunst, Bradley Pierce...
2    [Walte

## 6. Funcion para normalizar las listas

In [17]:
### Convierte campos tipo lista a texto normalizado.

def normalize_list_field(value):

    if value is None:
        return ""

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, list):
        return " ".join(
            str(item)
            .lower()
            .strip()
            .replace(" ", "_")
            for item in value
            if item is not None
        )

    if pd.isna(value):
        return ""

    return (
        str(value)
        .lower()
        .strip()
    )

## 7. Función que normaliza los campos simples

In [18]:
### Normaliza campos textuales simples.
    
def normalize_text(value):

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return (
        str(value)
        .lower()
        .strip()
    )

## 8.Procesamos datos

#### 8.1. Creamos una copia

In [20]:
content_model = content.copy()

#### 8.2. Procesamos generos

In [21]:
content_model["genres_text"] = (
    content_model["genres"]
    .apply(normalize_list_field)
)

#### 8.3. Procesamos TMDB genres

In [22]:
content_model["tmdb_genres_text"] = (
    content_model["tmdb_genres"]
    .apply(normalize_list_field)
)

#### 8.4. Procesamos keywords

In [23]:
content_model["keywords_text"] = (
    content_model["keywords"]
    .apply(normalize_list_field)
)

#### 8.5. Procesamos actores

In [24]:
content_model["actors_text"] = (
    content_model["actors"]
    .apply(normalize_list_field)
)

#### 8.6. Procesamos directores

In [25]:
content_model["director_text"] = (
    content_model["director"]
    .apply(normalize_text)
    .str.replace(
        " ",
        "_",
        regex=False,
    )
)

#### 8.7. Procesamos sinopsis 

In [26]:
content_model["overview_text"] = (
    content_model["overview"]
    .apply(normalize_text)
)

#### 8.8. Procesamos idioma

In [27]:
content_model["language_text"] = (
    content_model["original_language"]
    .apply(normalize_text)
)

## 9. Crear el content_soup

In [28]:
### Se combinan todas las caracteristicas en una sola representacion

content_model["content_soup"] = (
    content_model["genres_text"]
    + " "
    + content_model["tmdb_genres_text"]
    + " "
    + content_model["keywords_text"]
    + " "
    + content_model["director_text"]
    + " "
    + content_model["actors_text"]
    + " "
    + content_model["language_text"]
    + " "
    + content_model["overview_text"]
)

content_model[
    [
        "movie_id",
        "title",
        "content_soup",
    ]
].head()

,movie_id,title,content_soup
0,1,Toy Story,adventure|animation|children|comedy|fantasy fa...
1,2,Jumanji,adventure|children|fantasy aventura fantasía f...
2,3,Discordias a la carta,comedy|romance romance comedia fishing sequel ...
3,4,Esperando un respiro,comedy|drama|romance comedia drama romance bas...
4,5,Vuelve el padre de la novia (Ahora también abu...,comedy comedia familia daughter baby parent_ch...


## 10. Verificamos peliculas sin informacion

In [29]:
sin_contenido = (
    content_model["content_soup"]
    .str.strip()
    .eq("")
)

print(
    "Películas sin contenido:",
    sin_contenido.sum()
)

Películas sin contenido: 0


## 11. Creamos TF-IDF

In [30]:
### Convertimos el texto en vectores numericos

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50_000,
    min_df=2,
    ngram_range=(1, 2),
)

#### 11.1. Entrenamos

In [31]:
tfidf_matrix = tfidf.fit_transform(
    content_model["content_soup"]
)

#### 11.2. Verificamos

In [34]:
print(
    "Matriz TF-IDF:",
    tfidf_matrix.shape
)

print(
    "Elementos almacenados:",
    tfidf_matrix.nnz
)

Matriz TF-IDF: (87585, 275)
Elementos almacenados: 245709


#### TF-IDF

TF-IDF transforma el contenido textual de cada película en un vector numérico.

- TF (Term Frequency) mide la frecuencia de una palabra dentro de una película.
- IDF (Inverse Document Frequency) reduce el peso de términos demasiado comunes.
- Como resultado, las palabras más representativas de cada película tienen mayor influencia.

Esto permite comparar películas matemáticamente según sus características.

## 12. Contruir modelo de vecinos 

In [35]:
nearest_neighbors = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

#### 12.1. Entrenamos

In [36]:
nearest_neighbors.fit(
    tfidf_matrix
)

,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


## 12. Creamos Indice pelicula --> Posición 

In [37]:
### Convertimos movie_id a posicion de la Matriz

movie_id_to_idx = {
    movie_id: idx
    for idx, movie_id
    in enumerate(
        content_model["movie_id"]
    )
}

In [39]:
### por titulo

title_to_idx = {
    title.lower(): idx
    for idx, title
    in enumerate(
        content_model["title"]
    )
    if pd.notna(title)
}

## 13. Función de peliculas similares por ID

In [43]:
###   Devuelve las N películas más similares una película determinada.
def recommend_similar_movies(
    movie_id,
    n=10,
):

    if movie_id not in movie_id_to_idx:
        raise ValueError(
            "movie_id no encontrado."
        )

    movie_idx = movie_id_to_idx[
        movie_id
    ]

    distances, indices = (
        nearest_neighbors.kneighbors(
            tfidf_matrix[movie_idx],
            n_neighbors=n + 1,
        )
    )

    distances = distances.flatten()
    indices = indices.flatten()

    resultados = []

    for idx, distance in zip(
        indices,
        distances,
    ):

        # Excluir la misma película
        if idx == movie_idx:
            continue

        similarity = 1 - distance

        resultados.append({
            "movie_id": content_model.iloc[
                idx
            ]["movie_id"],

            "title": content_model.iloc[
                idx
            ]["title"],

            "similarity_score": similarity,
        })

    resultado_df = pd.DataFrame(
        resultados
    )

    return resultado_df.head(n)

## 14. Probamos la recomendacion por pelicula 

In [44]:
content_model[
    content_model["title"]
    .str.contains(
        "Toy Story",
        case=False,
        na=False,
    )
][
    [
        "movie_id",
        "title",
    ]
].head(10)

,movie_id,title
0,1,Toy Story
3021,3114,Toy Story 2 (1999)
14815,78499,Toy Story 3 (2010)
20505,106022,Toy Story of Terror (2013)
22646,115875,Toy Story Toons: Hawaiian Vacation (2011)
22647,115879,Toy Story Toons: Small Fry (2011)
24088,120468,Toy Story Toons: Partysaurus Rex (2012)
24090,120474,Toy Story That Time Forgot (2014)
60708,201588,Toy Story 4 (2019)


In [46]:
movie_test = 1

recommend_similar_movies(
    movie_id=movie_test,
    n=10,
)

,movie_id,title,similarity_score
0,3,Discordias a la carta,0.700259
1,6,Heat,0.687359
2,7,Sabrina (y sus amores),0.674056
3,4,Esperando un respiro,0.649000
4,2,Jumanji,0.543458
5,5,Vuelve el padre de la novia (Ahora también abu...,0.511328
6,10,Goldeneye,0.481801
7,9,Muerte súbita,0.460263
8,8,Tom y Huck,0.422228
9,181601,Olaf's Frozen Adventure (2017),0.142296


## 15. Agregar Match %


In [54]:
#### Esto mide la similitud de contenido, no la probabilidad de gusto

similar_movies = recommend_similar_movies(
    movie_id=movie_test,
    n=10,
)

similar_movies[
    "similarity_percent"
] = (
    similar_movies[
        "similarity_score"
    ]
    * 100
).round(1)

similar_movies[
    [
        "title",
        "similarity_score",
        "similarity_percent",
    ]
]

,title,similarity_score,similarity_percent
0,Discordias a la carta,0.700259,70.0
1,Heat,0.687359,68.7
2,Sabrina (y sus amores),0.674056,67.4
3,Esperando un respiro,0.649000,64.9
4,Jumanji,0.543458,54.3
5,Vuelve el padre de la novia (Ahora también abu...,0.511328,51.1
6,Goldeneye,0.481801,48.2
7,Muerte súbita,0.460263,46.0
8,Tom y Huck,0.422228,42.2
9,Olaf's Frozen Adventure (2017),0.142296,14.2


Ahora se personalizan las opciones:
- Si te gustó esta pelicula, estas son similares

O en el caso de CineMatch 
- Basandonos en las peliculas que te gustaron, estas sin tus recomendaciones

## 16. Cargar Ratings 

In [52]:
from src.utils.paths import (
    GOLD_FACTS_DIR,
)

fact_rating = pd.read_parquet(
    GOLD_FACTS_DIR
    / "fact_rating"
)

print(
    fact_rating.shape
)

fact_rating.head()

(32000204, 5)


,user_id,movie_id,date_key,rating,rated_at
0,1,17,19991203,4.0,1999-12-03 19:24:37+00:00
1,1,25,19991203,1.0,1999-12-03 19:43:48+00:00
2,1,29,19991122,2.0,1999-11-22 00:36:16+00:00
3,1,30,19991203,5.0,1999-12-03 19:24:37+00:00
4,1,32,19991122,5.0,1999-11-22 00:00:58+00:00


#### 16.1. Seleccionamos un usuario de prueba

In [ ]:
### Buscamos usuarios con suficientes datos positivos

positive_ratings = (
    fact_rating[
        fact_rating["rating"] >= 4.0
    ]
)

In [ ]:
### Mostramos los usuarios con mayor cantidad

positive_counts = (
    positive_ratings
    .groupby("user_id")
    .size()
    .sort_values(
        ascending=False
    )
)

positive_counts.head()

user_id
87007     5525
175325    4852
23783     3852
103925    3405
134403    3215
dtype: int64

In [ ]:
### Seleccionamos el  tiene mas

test_user_id = (
    positive_counts.index[0]
)

print(
    "Usuario:",
    test_user_id
)

Usuario: 87007


#### 16.1. Mostramos las peliculas qeu le gustaron 

In [62]:
liked_movies = (
    positive_ratings[
        positive_ratings["user_id"]
        == test_user_id
    ]
)

### Unimso ocn los titulos
liked_movies = liked_movies.merge(
    content_model[
        [
            "movie_id",
            "title",
        ]
    ],
    on="movie_id",
    how="left",
)

### Mostramos
liked_movies[
    [
        "movie_id",
        "title",
        "rating",
    ]
].head(20)

,movie_id,title,rating
0,111,Taxi Driver (1976),5.0
1,260,Star Wars: Episode IV - A New Hope (1977),5.0
2,750,Dr. Strangelove or: How I Learned to Stop Worr...,5.0
3,858,"Godfather, The (1972)",5.0
4,901,Funny Face (1957),5.0
5,902,Breakfast at Tiffany's (1961),5.0
6,911,Charade (1963),5.0
7,912,Casablanca (1942),5.0
8,917,"Little Princess, The (1939)",5.0
9,919,"Wizard of Oz, The (1939)",5.0


## 17. Construimos el perfilb Content-Based del usuario

In [ ]:
### Tomamos los vectores TF-TDF de peliculas que valoró con 4 o mas

liked_movie_ids = (
    liked_movies["movie_id"]
    .dropna()
    .tolist()
)

In [68]:
### Con solo los IDs disponibles en nuestro modelo

liked_indices = [
    movie_id_to_idx[movie_id]
    for movie_id in liked_movie_ids
    if movie_id in movie_id_to_idx
]

In [69]:
### Calculamos el vector promedio 

user_profile = (
    tfidf_matrix[
        liked_indices
    ]
    .mean(axis=0)
)

In [72]:
### Lo convertimos a array

user_profile = np.asarray(
    user_profile
)

## 18. Buscamos peliculas similares al perfil

In [73]:
distances, indices = (
    nearest_neighbors.kneighbors(
        user_profile,
        n_neighbors=100,
    )
)

In [75]:
### Excluimos las que ya vio

seen_movies = set(
    fact_rating.loc[
        fact_rating["user_id"]
        == test_user_id,
        "movie_id",
    ]
)

In [76]:
### Generamos las recomendaciones

recommendations = []

for idx, distance in zip(
    indices.flatten(),
    distances.flatten(),
):

    movie_id = content_model.iloc[
        idx
    ]["movie_id"]

    if movie_id in seen_movies:
        continue

    similarity = 1 - distance

    recommendations.append({
        "movie_id": movie_id,
        "title": content_model.iloc[
            idx
        ]["title"],
        "content_score": similarity,
    })

    if len(recommendations) == 10:
        break

In [79]:
### Creamos un DataFrame

content_top_10 = pd.DataFrame(
    recommendations
)

content_top_10

,movie_id,title,content_score
0,26,Othello (1995),0.589973
1,61206,In the City of Sylvia (En la ciudad de Sylvia)...,0.589973
2,60758,Brideshead Revisited (2008),0.589973
3,60654,"Last Mistress, The (vieille maîtresse, Une) (2...",0.589973
4,207431,They Don't Wear Black Tie (1980),0.589973
5,207427,The boundaries (1995),0.589973
6,207505,Nightmares (1979),0.589973
7,61009,Inside Paris (Dans Paris) (2006),0.589973
8,60943,Frozen River (2008),0.589973
9,60930,Frozen City (Valkoinen kaupunki) (2006),0.589973


## 19. Content Match %

In [ ]:
### Content-based del usuario

content_top_10[
    "content_match_percent"
] = (
    content_top_10[
        "content_score"
    ]
    * 100
).round(1)

content_top_10[
    [
        "title",
        "content_score",
        "content_match_percent",
    ]
]

,title,content_score,content_match_percent
0,Othello (1995),0.589973,59.0
1,In the City of Sylvia (En la ciudad de Sylvia)...,0.589973,59.0
2,Brideshead Revisited (2008),0.589973,59.0
3,"Last Mistress, The (vieille maîtresse, Une) (2...",0.589973,59.0
4,They Don't Wear Black Tie (1980),0.589973,59.0
5,The boundaries (1995),0.589973,59.0
6,Nightmares (1979),0.589973,59.0
7,Inside Paris (Dans Paris) (2006),0.589973,59.0
8,Frozen River (2008),0.589973,59.0
9,Frozen City (Valkoinen kaupunki) (2006),0.589973,59.0


## 20. Funcion reutilizable por usuario

In [ ]:
### Funcion que genera recomendaciones personalizadas por un usuario completa

def recommend_content_for_user(
    user_id,
    n=10,
    min_rating=4.0,
):

    user_ratings = (
        fact_rating[
            fact_rating["user_id"]
            == user_id
        ]
    )

    liked = user_ratings[
        user_ratings["rating"]
        >= min_rating
    ]

    liked_ids = liked[
        "movie_id"
    ].tolist()

    liked_indices = [
        movie_id_to_idx[movie_id]
        for movie_id in liked_ids
        if movie_id in movie_id_to_idx
    ]

    if not liked_indices:
        raise ValueError(
            "El usuario no tiene suficientes "
            "películas positivas para crear perfil."
        )

    profile = (
        tfidf_matrix[
            liked_indices
        ]
        .mean(axis=0)
    )

    profile = np.asarray(
        profile
    )

    # Buscamos más candidatos que el Top-N final
    candidate_count = min(
        max(n * 20, 100),
        len(content_model)
    )

    distances, indices = (
        nearest_neighbors.kneighbors(
            profile,
            n_neighbors=candidate_count,
        )
    )

    seen_movies = set(
        user_ratings[
            "movie_id"
        ]
    )

    recommendations = []

    for idx, distance in zip(
        indices.flatten(),
        distances.flatten(),
    ):

        movie_id = (
            content_model.iloc[
                idx
            ]["movie_id"]
        )

        if movie_id in seen_movies:
            continue

        score = 1 - distance

        recommendations.append({
            "movie_id": movie_id,
            "title": (
                content_model.iloc[
                    idx
                ]["title"]
            ),
            "content_score": score,
        })

        if len(recommendations) >= n:
            break

    df = pd.DataFrame(
        recommendations
    )

    if not df.empty:
        df[
            "content_match_percent"
        ] = (
            df["content_score"]
            * 100
        ).round(1)

    return df

recommend_content_for_user(
    test_user_id,
    n=10,
)

,movie_id,title,content_score,content_match_percent
0,26,Othello (1995),0.589973,59.0
1,61206,In the City of Sylvia (En la ciudad de Sylvia)...,0.589973,59.0
2,60758,Brideshead Revisited (2008),0.589973,59.0
3,60654,"Last Mistress, The (vieille maîtresse, Une) (2...",0.589973,59.0
4,207431,They Don't Wear Black Tie (1980),0.589973,59.0
5,207427,The boundaries (1995),0.589973,59.0
6,207505,Nightmares (1979),0.589973,59.0
7,61009,Inside Paris (Dans Paris) (2006),0.589973,59.0
8,60943,Frozen River (2008),0.589973,59.0
9,60930,Frozen City (Valkoinen kaupunki) (2006),0.589973,59.0


## 21. Validaciones del modelo

In [84]:
print(
    "Películas vectorizadas:",
    tfidf_matrix.shape[0]
)

print(
    "Features TF-IDF:",
    tfidf_matrix.shape[1]
)

print(
    "Usuario prueba:",
    test_user_id
)

print(
    "Películas positivas:",
    len(liked_indices)
)

Películas vectorizadas: 87585
Features TF-IDF: 275
Usuario prueba: 87007
Películas positivas: 5525


## Conclusiones

Se desarrolló un modelo de recomendación Content-Based para CineMatch.

Las características de las películas fueron representadas mediante TF-IDF utilizando géneros, keywords, director, actores, idioma y sinopsis. Para evitar construir una matriz completa de similitud entre las 87 mil películas, se utilizó NearestNeighbors con distancia coseno sobre una matriz dispersa.

Además de recomendar películas similares a un título específico, se construyó un perfil de usuario utilizando las películas calificadas positivamente. Esto permite generar recomendaciones personalizadas basadas en los contenidos que históricamente han sido de su preferencia.

El score obtenido representa similitud de contenido y posteriormente será combinado con el score del modelo colaborativo para construir el recomendador híbrido de CineMatch.